# 🔀 Day 03a: Sharding & Replication — Scaling Databases Horizontally

---

## 🎯 What You'll Master Today
- Horizontal vs vertical scaling
- Sharding strategies (hash, range, directory)
- Consistent hashing
- Replication (master-slave, master-master, multi-Paxos)
- Quorum reads/writes

---

### 🎭 The Analogy

**Sharding** = splitting a library across multiple buildings. Building A has books A-M, Building B has N-Z.  
**Replication** = making photocopies. Each building has the SAME copy of every book.

```
╔══════════════════════════════════════════════════════════════════════╗
║               SCALING STRATEGIES                                     ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  VERTICAL SCALING (Scale Up)      HORIZONTAL SCALING (Scale Out)    ║
║  ┌──────────┐                     ┌────┐ ┌────┐ ┌────┐ ┌────┐    ║
║  │          │                     │ S1 │ │ S2 │ │ S3 │ │ S4 │    ║
║  │  BIGGER  │                     └────┘ └────┘ └────┘ └────┘    ║
║  │  SERVER  │                     Add more machines               ║
║  │  (+RAM)  │                                                      ║
║  │  (+CPU)  │                     ✅ Unlimited scale               ║
║  └──────────┘                     ✅ Fault tolerant                 ║
║  ❌ Hardware limits               ❌ Complex (sharding/replication) ║
║  ❌ Single point of failure       ❌ Distributed transactions      ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Sharding Strategies
# ============================================

import hashlib

class HashSharder:
    """Simple hash-based sharding."""
    def __init__(self, n_shards):
        self.n_shards = n_shards
        self.shards = {i: [] for i in range(n_shards)}
    
    def get_shard(self, key):
        h = int(hashlib.md5(str(key).encode()).hexdigest(), 16)
        return h % self.n_shards
    
    def insert(self, key, value):
        shard = self.get_shard(key)
        self.shards[shard].append((key, value))
        return shard

class RangeSharder:
    """Range-based sharding."""
    def __init__(self, ranges):
        self.ranges = ranges  # [(start, end, shard_name), ...]
    
    def get_shard(self, key):
        for start, end, name in self.ranges:
            if start <= key <= end:
                return name
        return "overflow"

# Hash sharding demo
print("  Hash-Based Sharding (3 shards):\n")
hs = HashSharder(3)
users = [("user_1", "Alice"), ("user_2", "Bob"), ("user_3", "Charlie"),
         ("user_4", "Diana"), ("user_5", "Eve"), ("user_6", "Frank")]

for key, val in users:
    shard = hs.insert(key, val)
    print(f"    {key} ({val}) → Shard {shard}")

print("\n  Shard distribution:")
for s, data in hs.shards.items():
    print(f"    Shard {s}: {[v for _, v in data]}")

# Range sharding demo
print("\n  Range-Based Sharding:\n")
rs = RangeSharder([(1, 1000, "Shard-A"), (1001, 2000, "Shard-B"), (2001, 3000, "Shard-C")])
for uid in [42, 999, 1500, 2500]:
    print(f"    user_id={uid} → {rs.get_shard(uid)}")

print("\n  Hash: even distribution, no range queries")
print("  Range: supports range queries, risk of hotspots")

In [ ]:
# ============================================
# 2. Consistent Hashing
# ============================================

import bisect

class ConsistentHash:
    def __init__(self, n_virtual=150):
        self.n_virtual = n_virtual
        self.ring = []          # sorted list of hash positions
        self.ring_map = {}      # hash_pos → node_name
        self.nodes = set()
    
    def _hash(self, key):
        return int(hashlib.md5(key.encode()).hexdigest(), 16) % 360
    
    def add_node(self, node):
        self.nodes.add(node)
        for i in range(self.n_virtual):
            h = self._hash(f"{node}:{i}")
            self.ring.append(h)
            self.ring_map[h] = node
        self.ring.sort()
    
    def remove_node(self, node):
        self.nodes.discard(node)
        to_remove = [h for h, n in self.ring_map.items() if n == node]
        for h in to_remove:
            self.ring.remove(h)
            del self.ring_map[h]
    
    def get_node(self, key):
        if not self.ring:
            return None
        h = self._hash(key)
        idx = bisect.bisect_right(self.ring, h) % len(self.ring)
        return self.ring_map[self.ring[idx]]

ch = ConsistentHash(n_virtual=3)  # small for demo
for node in ["Node-A", "Node-B", "Node-C"]:
    ch.add_node(node)

print("  Consistent Hashing — 3 Nodes:\n")
keys = ["user:1", "user:2", "user:3", "order:100", "product:42"]
mapping1 = {}
for key in keys:
    node = ch.get_node(key)
    mapping1[key] = node
    print(f"    {key:15} → {node}")

# Add a node — only ~1/n keys move
print("\n  Adding Node-D...")
ch.add_node("Node-D")
moved = 0
for key in keys:
    node = ch.get_node(key)
    changed = " ← MOVED" if node != mapping1[key] else ""
    if changed: moved += 1
    print(f"    {key:15} → {node}{changed}")

print(f"\n  Keys moved: {moved}/{len(keys)} (~{moved/len(keys)*100:.0f}%)")
print("  Without consistent hashing: ALL keys would remap!")
print("  With consistent hashing: only ~K/N keys move (K=keys, N=nodes)")

In [ ]:
# ============================================
# 3. Replication Strategies
# ============================================

import threading

class ReplicaNode:
    def __init__(self, name, role="replica"):
        self.name = name
        self.role = role
        self.data = {}
        self.log = []       # WAL for replication
    
    def write(self, key, value, lsn):
        self.data[key] = value
        self.log.append((lsn, key, value))

class MasterSlaveCluster:
    def __init__(self, n_replicas=2, mode="sync"):
        self.master = ReplicaNode("Master", "master")
        self.replicas = [ReplicaNode(f"Replica-{i}") for i in range(n_replicas)]
        self.mode = mode  # sync or async
        self.lsn = 0
    
    def write(self, key, value):
        self.lsn += 1
        self.master.write(key, value, self.lsn)
        print(f"    ✅ Master: wrote {key}={value} (LSN={self.lsn})")
        
        if self.mode == "sync":
            for r in self.replicas:
                r.write(key, value, self.lsn)
                print(f"    ✅ {r.name}: replicated (sync) LSN={self.lsn}")
        else:  # async
            print(f"    ⏳ Replicas: will replicate asynchronously (lag possible)")
    
    def read(self, key, from_replica=False):
        if from_replica:
            replica = self.replicas[0]
            val = replica.data.get(key, "NOT FOUND")
            print(f"    READ from {replica.name}: {key}={val}")
        else:
            val = self.master.data.get(key, "NOT FOUND")
            print(f"    READ from Master: {key}={val}")

# Synchronous replication
print("  Master-Slave: Synchronous Replication")
print("  " + "─" * 50)
sync_cluster = MasterSlaveCluster(mode="sync")
sync_cluster.write("balance", 1000)
sync_cluster.read("balance", from_replica=True)
print("  Pro: Strong consistency. Con: Higher latency.\n")

# Asynchronous replication
print("  Master-Slave: Asynchronous Replication")
print("  " + "─" * 50)
async_cluster = MasterSlaveCluster(mode="async")
async_cluster.write("balance", 1000)
print("  Pro: Low latency. Con: Stale reads from replicas.")

In [ ]:
# ============================================
# 4. Replication Topologies Comparison
# ============================================

print("""
  ┌────────────────────────────────────────────────────────────────────┐
  │               REPLICATION TOPOLOGIES                               │
  ├────────────────────────────────────────────────────────────────────┤
  │                                                                    │
  │  1. MASTER-SLAVE (Primary-Secondary)                              │
  │     ┌────────┐  sync/async   ┌────────┐                          │
  │     │ Master │ ────────────→ │ Slave  │                          │
  │     │  (R/W) │ ────────────→ │ (Read) │                          │
  │     └────────┘               └────────┘                          │
  │     Writes → Master only. Reads → any replica.                   │
  │     Used by: MySQL, PostgreSQL, MongoDB, Redis                   │
  │                                                                    │
  │  2. MASTER-MASTER (Multi-Primary)                                 │
  │     ┌────────┐  ←──────────→  ┌────────┐                         │
  │     │Master-1│  sync both     │Master-2│                         │
  │     │  (R/W) │  ←──────────→  │  (R/W) │                         │
  │     └────────┘                └────────┘                         │
  │     Writes → any node. Conflict resolution needed!               │
  │     Used by: CouchDB, Galera Cluster                             │
  │                                                                    │
  │  3. LEADERLESS (Dynamo-style)                                     │
  │     ┌────┐  ┌────┐  ┌────┐                                      │
  │     │ N1 │  │ N2 │  │ N3 │  All peers, no master                │
  │     └────┘  └────┘  └────┘                                      │
  │     Client writes to W nodes, reads from R nodes                 │
  │     Used by: Cassandra, DynamoDB, Riak                           │
  │                                                                    │
  ├────────────────────────────────────────────────────────────────────┤
  │ Topology     │ Consistency │ Availability │ Write perf │ Complex │
  ├──────────────┼─────────────┼──────────────┼────────────┼─────────┤
  │ Master-slave │ Strong      │ Read HA      │ 1 node     │ Low     │
  │ Master-master│ Eventual    │ Full HA      │ N nodes    │ High    │
  │ Leaderless   │ Tunable     │ Full HA      │ W nodes    │ Medium  │
  └────────────────────────────────────────────────────────────────────┘
""")

In [ ]:
# ============================================
# 5. Sharding Challenges
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════╗
  ║            SHARDING CHALLENGES (Interview Gold!)               ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║                                                                  ║
  ║  1. CROSS-SHARD JOINS                                           ║
  ║     Problem: SELECT * FROM orders JOIN users                    ║
  ║             users on Shard A, orders on Shard B                 ║
  ║     Fix: Denormalize (embed user_name in orders)                ║
  ║                                                                  ║
  ║  2. HOTSPOTS                                                    ║
  ║     Problem: Celebrity user → all traffic to 1 shard            ║
  ║     Fix: Add random suffix to key, split hot partition          ║
  ║                                                                  ║
  ║  3. REBALANCING                                                 ║
  ║     Problem: Adding/removing shards → data migration            ║
  ║     Fix: Consistent hashing, virtual nodes                      ║
  ║                                                                  ║
  ║  4. DISTRIBUTED TRANSACTIONS                                    ║
  ║     Problem: Transfer between accounts on different shards      ║
  ║     Fix: 2PC (slow), Saga pattern (eventual), avoid if possible ║
  ║                                                                  ║
  ║  5. AUTO-INCREMENT IDs                                          ║
  ║     Problem: Each shard generates its own IDs → collisions      ║
  ║     Fix: UUID, Snowflake ID, ID service (Twitter/Instagram)     ║
  ║                                                                  ║
  ╚══════════════════════════════════════════════════════════════════╝
""")

# Snowflake ID demo
import time, threading

class SnowflakeID:
    def __init__(self, machine_id):
        self.machine_id = machine_id & 0x3FF  # 10 bits
        self.sequence = 0
        self.last_ts = 0
        self.epoch = 1704067200000  # 2024-01-01
    
    def generate(self):
        ts = int(time.time() * 1000) - self.epoch
        if ts == self.last_ts:
            self.sequence = (self.sequence + 1) & 0xFFF  # 12 bits
        else:
            self.sequence = 0
        self.last_ts = ts
        # 41-bit timestamp | 10-bit machine | 12-bit sequence
        return (ts << 22) | (self.machine_id << 12) | self.sequence

sf = SnowflakeID(machine_id=1)
print("  Snowflake IDs (globally unique, sortable):")
for _ in range(5):
    print(f"    {sf.generate()}")
print("  Bits: [41 timestamp | 10 machine | 12 sequence] = 63 bits")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Horizontal vs vertical scaling? | Vertical: bigger machine (limits). Horizontal: more machines (sharding) |
| 2 | Hash vs range sharding? | Hash: even distribution, no range queries. Range: range queries, hotspot risk |
| 3 | What is consistent hashing? | Ring-based. Adding node moves only K/N keys. Virtual nodes for balance |
| 4 | Sync vs async replication? | Sync: strong consistency, higher latency. Async: low latency, stale reads |
| 5 | How to handle cross-shard joins? | Denormalize, application-level joins, or co-locate related data on same shard |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Sharding: split data across nodes (hash/range)       │
## │  • Consistent hashing: minimal reshuffling on change    │
## │  • Replication: copies for HA + read scaling            │
## │  • Master-slave: simple, read-scale                     │
## │  • Leaderless: W+R>N for consistency (Dynamo-style)     │
## │  • Snowflake IDs: globally unique, time-sortable       │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Database Design & Interview Practice** — End-to-end schema design